# Task 1: News Topic Classifier Using BERT
This notebook fine-tunes `bert-base-uncased` on the AG News dataset using Hugging Face Transformers.

## 1. Install Dependencies

In [ ]:
!pip -q install transformers datasets evaluate accelerate gradio scikit-learn

## 2. Import Libraries

In [ ]:
from datasets import load_dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
TrainingArguments, Trainer)
import evaluate
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix


## 3. Load Dataset

In [ ]:
dataset=load_dataset("ag_news")
dataset

## 4. Tokenization

In [ ]:
checkpoint="bert-base-uncased"
tokenizer=AutoTokenizer.from_pretrained(checkpoint)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=128)

tokenized=dataset.map(tokenize, batched=True)
tokenized=tokenized.rename_column("label","labels")
tokenized.set_format("torch", columns=["input_ids","attention_mask","labels"])


## 5. Load Model

In [ ]:
model=AutoModelForSequenceClassification.from_pretrained(checkpoint,num_labels=4)
accuracy=evaluate.load("accuracy")
f1=evaluate.load("f1")

def compute_metrics(eval_pred):
    logits,labels=eval_pred
    preds=np.argmax(logits,axis=-1)
    return {
        "accuracy":accuracy.compute(predictions=preds,references=labels)["accuracy"],
        "f1":f1.compute(predictions=preds,references=labels,average="weighted")["f1"]
    }


## 6. Train

In [ ]:
args=TrainingArguments(
    output_dir="bert-news",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=100,
    load_best_model_at_end=True,
    report_to="none"
)

trainer=Trainer(
    model=model,
    args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

trainer.train()


## 7. Evaluation

In [ ]:
results=trainer.evaluate()
print(results)

pred=trainer.predict(tokenized["test"])
preds=np.argmax(pred.predictions,axis=-1)
print(classification_report(pred.label_ids,preds))
print(confusion_matrix(pred.label_ids,preds))


## 8. Save Model

In [ ]:
trainer.save_model("bert_news_model")
tokenizer.save_pretrained("bert_news_model")


## 9. Inference

In [ ]:
labels=["World","Sports","Business","Sci/Tech"]

def predict_news(text):
    inputs=tokenizer(text,return_tensors="pt",truncation=True,padding=True,max_length=128)
    outputs=model(**inputs)
    pred=outputs.logits.argmax(-1).item()
    return labels[pred]

print(predict_news("Apple unveils new AI-powered iPhone"))


## 10. Gradio App

In [ ]:
import gradio as gr
demo=gr.Interface(fn=predict_news,
                  inputs=gr.Textbox(lines=3,label="Headline"),
                  outputs="text",
                  title="BERT News Topic Classifier")
demo.launch()
